Explore workbook creation, loading, configuration comparison, and metadata upserts using temporary files and a mock SQL client.


In [ ]:
from setup.nbs.inspect_tools import show, show_methods
from pair_recon.core.metadata.workbook import create_pair_workbook, read_pair_workbook
from pair_recon.core.metadata.workbook import pair_configuration_signature, changed_pair_configurations
from pair_recon.core.metadata.workbook import upsert_pair_configuration
from pair_recon.core.metadata.configuration import parse_key_columns, build_pair_config

show(create_pair_workbook, read_pair_workbook, pair_configuration_signature)
show(changed_pair_configurations, upsert_pair_configuration, parse_key_columns, build_pair_config)


In [ ]:
from pair_recon.core.metadata.configuration import parse_key_columns

# key 类型在 workbook 边界解析，运行时使用规范化结构
parse_key_columns(['ORDER_ID[numeric]', 'EFF_DT[date]'])


## Create and read a workbook

The workbook is created in a temporary directory. Reading the empty template returns an empty list.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "pairs.xlsx"
    created = create_pair_workbook(path)
    print("created:", created.name, created.exists())
    print("rows:", read_pair_workbook(path))


## Parse key columns

A list of typed key specifications becomes a list of names and a type mapping.


In [ ]:
keys = ["ORDER_ID[numeric]", "EFF_DT[date]"]
print("input:", keys)
print("output:", parse_key_columns(keys))


## Build and compare a pair configuration

This small legacy-style row supplies `config_json`. The comparison normalizes the row into a signature before detecting changes.


In [ ]:
from pprint import pprint

row = {"pair_name": "ORDERS", "enabled": True, "key_columns": ["ORDER_ID"], "config_json": {"left": {}, "right": {}}}
print("input row:")
pprint(row)
print("built config:")
pprint(build_pair_config(row))
print("signature:")
pprint(pair_configuration_signature(row))
print("new pair:", [item["pair_name"] for item in changed_pair_configurations([row], [])])
print("unchanged:", changed_pair_configurations([row], [row]))


## Upsert with a mock SQL client

The client records the SQL and parameters; no database is contacted. The input row is the same one used above.


In [ ]:
from unittest.mock import Mock

sql = Mock()
result = upsert_pair_configuration(sql, "demo.recon.pair_recon_pair_meta", [row])
print("result:", result)
print("SQL calls:", sql.execute.call_count)
for call in sql.execute.call_args_list:
    statement, parameters = call.args
    print("target:", statement.splitlines()[0])
    print("pair_name:", parameters["pair_name"])
